# Day 3 - Multi-Model Comparison, Tuning and Bias-Variance (RetailMind AI)

**Goal:** tune each candidate model with `GridSearchCV` + cross-validation, check whether each one underfits (high bias) or overfits (high variance), and lock the best model as `churn_v3.joblib`.

## 1. Setup

In [1]:
import sys
sys.path.append("..")

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from src.config import MODEL_PATH
from src.data_loader import load_data, split_features_target
from src.preprocessing import build_preprocessor
from src.evaluate import get_metrics

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 2. Same data and split as before

In [2]:
df = load_data()
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

Train rows: 5634 | Test rows: 1409


## 3. Define a search grid for each candidate
Each model gets a small grid of settings to try. All use `class_weight="balanced"` (from Day 2) so recall stays a priority.

- **Logistic Regression:** `C` = regularisation strength (smaller = simpler model).
- **Decision Tree:** `max_depth` and `min_samples_leaf` limit how much the tree can memorise.
- **Random Forest:** number of trees, depth and leaf size.
- **SVM:** `C` and `gamma` control how flexible the boundary is.

In [3]:
candidates = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        {"model__C": [0.01, 0.1, 1, 10]},
    ),
    "Decision Tree": (
        DecisionTreeClassifier(class_weight="balanced", random_state=42),
        {"model__max_depth": [3, 5, 7, 10], "model__min_samples_leaf": [1, 10, 30]},
    ),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", random_state=42),
        {"model__n_estimators": [100, 200], "model__max_depth": [5, 8, None],
         "model__min_samples_leaf": [1, 5, 10]},
    ),
    "SVM": (
        SVC(class_weight="balanced", probability=True, random_state=42),
        {"model__C": [0.1, 1, 10], "model__gamma": ["scale", 0.01]},
    ),
}

## 4. Run GridSearchCV with 5-fold cross-validation
For every combination of settings, the training data is split into 5 parts; the model trains on 4 and is scored on the 5th, five times. We score **ROC-AUC** (picks the best model) and **recall** (our business focus), and keep the training scores too so we can check for overfitting.


In [4]:
searches = {}
for name, (model, grid) in candidates.items():
    pipe = Pipeline([("preprocess", build_preprocessor(X_train)), ("model", model)])
    search = GridSearchCV(
        pipe, grid, cv=5,
        scoring={"roc_auc": "roc_auc", "recall": "recall"},
        refit="roc_auc",              # the best settings are chosen by ROC-AUC
        return_train_score=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    print(name, "-> best settings:", search.best_params_)

Logistic Regression -> best settings: {'model__C': 10}
Decision Tree -> best settings: {'model__max_depth': 5, 'model__min_samples_leaf': 30}
Random Forest -> best settings: {'model__max_depth': 8, 'model__min_samples_leaf': 10, 'model__n_estimators': 200}
SVM -> best settings: {'model__C': 1, 'model__gamma': 0.01}


## 5. Bias vs variance: train score vs validation score
- **Train and validation both low** → high bias (underfitting): the model is too simple.
- **Train high, validation much lower** → high variance (overfitting): the model memorised the training data.

In [5]:
rows = {}
for name, search in searches.items():
    i = search.best_index_
    scores = search.cv_results_
    train_auc = scores["mean_train_roc_auc"][i]
    val_auc = scores["mean_test_roc_auc"][i]
    rows[name] = {
        "Train ROC-AUC": round(train_auc, 4),
        "Validation ROC-AUC": round(val_auc, 4),
        "Gap (train - val)": round(train_auc - val_auc, 4),
        "Validation Recall": round(scores["mean_test_recall"][i], 4),
    }

cv_table = pd.DataFrame(rows).T
cv_table

,Train ROC-AUC,Validation ROC-AUC,Gap (train - val),Validation Recall
Logistic Regression,0.8497,0.8455,0.0042,0.8027
Decision Tree,0.8537,0.8255,0.0282,0.7860
Random Forest,0.8890,0.8489,0.0401,0.7813
SVM,0.8490,0.8412,0.0078,0.8261


**Reading the table**
- **Logistic Regression:** train and validation scores are almost identical (gap ~0.004), so it has very low variance. It is a simple model, so it can't capture as much as the forest, but it is only slightly behind.
- **Decision Tree:** a moderate gap and the lowest validation score, so it is the weakest candidate even after limiting its depth.
- **Random Forest:** the highest validation score, but also the biggest gap (~0.04). That is a little *variance* (some overfitting), which is normal for forests and still acceptable.
- **SVM:** a small gap and the best validation recall, with a slightly lower validation ROC-AUC.

Overall, no model is badly underfitting or overfitting. The scores are all close together, which hints that the data itself (current features) is the limit, not the algorithm.

In [6]:
champion_name = cv_table["Validation ROC-AUC"].idxmax()
champion = searches[champion_name].best_estimator_
print("Champion (best validation ROC-AUC):", champion_name)
print("Settings:", searches[champion_name].best_params_)

Champion (best validation ROC-AUC): Random Forest
Settings: {'model__max_depth': 8, 'model__min_samples_leaf': 10, 'model__n_estimators': 200}


## 6. Learning curve of the champion
We train on 20%, 40% ... 100% of the training data and watch the train and validation scores. This tells us whether **more data**, **more features** or **more regularisation** is what we need.

In [7]:
sizes, train_scores, val_scores = learning_curve(
    champion, X_train, y_train,
    train_sizes=[0.2, 0.4, 0.6, 0.8, 1.0], cv=5, scoring="roc_auc"
)

curve = pd.DataFrame({
    "Training rows": sizes,
    "Train ROC-AUC": train_scores.mean(axis=1).round(4),
    "Validation ROC-AUC": val_scores.mean(axis=1).round(4),
})
curve["Gap"] = (curve["Train ROC-AUC"] - curve["Validation ROC-AUC"]).round(4)
curve

,Training rows,Train ROC-AUC,Validation ROC-AUC,Gap
0,901,0.9169,0.8436,0.0733
1,1802,0.9081,0.8442,0.0639
2,2704,0.9029,0.8455,0.0574
3,3605,0.8970,0.8474,0.0496
4,4507,0.8889,0.8487,0.0402


**Reading the curve:** as the training data grows, the train score comes down (0.92 to 0.89) and the validation score creeps up (0.844 to 0.849), so the gap shrinks from about 0.07 to 0.04. The validation score improves only slightly even with 5 times more data, so **more rows of the same data probably won't help much**. Better gains are more likely from **new features** (that is Day 4) than from more data. The remaining gap also suggests a bit of extra regularisation could be tried.

## 7. Check on the hold-out test set
The test set was not used for tuning, so this is an honest final check.

In [8]:
test_rows = {name: get_metrics(search.best_estimator_, X_test, y_test)
             for name, search in searches.items()}
pd.DataFrame(test_rows).T

,Accuracy,Precision,Recall,F1,ROC-AUC
Logistic Regression,0.7395,0.5060,0.7834,0.6149,0.8405
Decision Tree,0.7559,0.5279,0.7594,0.6228,0.8327
Random Forest,0.7559,0.5268,0.7888,0.6317,0.8432
SVM,0.7090,0.4721,0.8155,0.5980,0.8338


## 8. Lock and version the champion

In [9]:
joblib.dump(champion, MODEL_PATH.parent / "churn_v3.joblib")
print("Saved churn_v3.joblib")

Saved churn_v3.joblib
